# 🌲 Random Forest From Scratch
### Built on First Principles with NumPy

---

We build everything bottom-up:

1. **Information Theory** — Entropy, Gini Impurity, Information Gain
2. **Decision Tree** — splitting logic, recursive tree building, prediction
3. **Bagging** — Bootstrap Aggregation, why it reduces variance
4. **Random Forest** — feature randomness on top of bagging
5. **Feature Importance** — what the forest actually learned
6. **Hyperparameter Tuning** — n_estimators, max_depth, max_features
7. **Regression Trees** — the same idea for continuous targets

---

> **No sklearn. No scipy. Just NumPy + math + intuition.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})

GREEN  = '#3fb950'
BLUE   = '#58a6ff'
ORANGE = '#f78166'
GOLD   = '#e3b341'
PURPLE = '#bc8cff'
TEAL   = '#39d353'

np.random.seed(42)
print('✅ Environment ready. Let\'s grow some trees.')

---
## Part 1 — Information Theory: Measuring Impurity

A decision tree splits data to make groups **purer** — more homogeneous.  
We need a number to measure how "mixed" a group is.

### Entropy
From information theory. Higher = more mixed.
$$H(S) = -\sum_{k} p_k \log_2(p_k)$$

### Gini Impurity
Probability of mislabelling a random element. Faster to compute.
$$G(S) = 1 - \sum_{k} p_k^2$$

### Information Gain
How much does a split reduce impurity?
$$IG = H(parent) - \frac{|L|}{|S|}H(L) - \frac{|R|}{|S|}H(R)$$

In [ ]:
def entropy(y):
    """Shannon entropy of label array y."""
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y.astype(int))
    probs  = counts[counts > 0] / len(y)
    return -np.sum(probs * np.log2(probs))

def gini(y):
    """Gini impurity of label array y."""
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y.astype(int))
    probs  = counts / len(y)
    return 1 - np.sum(probs ** 2)

def information_gain(y_parent, y_left, y_right, criterion='entropy'):
    """Reduction in impurity after a split."""
    fn  = entropy if criterion == 'entropy' else gini
    n   = len(y_parent)
    wl  = len(y_left)  / n
    wr  = len(y_right) / n
    return fn(y_parent) - wl * fn(y_left) - wr * fn(y_right)


# ── Visualise entropy & gini as p varies ─────────────────────────────
p_vals = np.linspace(0.001, 0.999, 300)
ent    = [-p*np.log2(p) - (1-p)*np.log2(1-p) for p in p_vals]
gin    = [2*p*(1-p) for p in p_vals]              # binary gini = 2p(1-p)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(p_vals, ent,  color=GREEN,  lw=2.5, label='Entropy H')
axes[0].plot(p_vals, gin,  color=ORANGE, lw=2.5, label='Gini G')
axes[0].axvline(0.5, color='#555', lw=1, linestyle=':')
axes[0].set_xlabel('p (fraction of class 1)')
axes[0].set_title('Impurity Measures\n(peak at p=0.5 = most mixed)', color=BLUE)
axes[0].legend(); axes[0].grid(True)

# Example groups
groups = [
    ('Pure\n[0,0,0,0]',       np.array([0,0,0,0])),
    ('Mixed\n[0,0,1,1]',      np.array([0,0,1,1])),
    ('Very Mixed\n[0,1,0,1]', np.array([0,1,0,1])),
    ('Mostly 1\n[1,1,1,0]',   np.array([1,1,1,0])),
]
names  = [g[0] for g in groups]
ent_v  = [entropy(g[1]) for g in groups]
gini_v = [gini(g[1])    for g in groups]

x = np.arange(len(groups))
axes[1].bar(x - 0.2, ent_v,  width=0.35, color=GREEN,  label='Entropy')
axes[1].bar(x + 0.2, gini_v, width=0.35, color=ORANGE, label='Gini')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, fontsize=8)
axes[1].set_title('Impurity of example groups', color=BLUE)
axes[1].legend(); axes[1].grid(True, axis='y')

# Information Gain example
parent = np.array([0,0,0,0,1,1,1,1])
split_good = (np.array([0,0,0,0]), np.array([1,1,1,1]))  # perfect
split_bad  = (np.array([0,0,1,1]), np.array([0,0,1,1]))  # no gain
ig_good = information_gain(parent, *split_good)
ig_bad  = information_gain(parent, *split_bad)

axes[2].bar(['Good split', 'Bad split'], [ig_good, ig_bad],
            color=[GREEN, ORANGE], width=0.4)
axes[2].set_ylabel('Information Gain')
axes[2].set_title(f'IG: good={ig_good:.2f}, bad={ig_bad:.2f}', color=BLUE)
axes[2].grid(True, axis='y')

plt.suptitle('Information Theory Foundations', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 2 — Decision Tree: The Core Building Block

A decision tree works by:
1. **Finding the best split** — which feature and threshold maximises Information Gain?
2. **Splitting the data** — partition into left (≤ threshold) and right (> threshold)
3. **Repeating recursively** — until a stopping condition (max depth, min samples, etc.)
4. **Predicting** — traverse the tree and return the majority class at the leaf

In [ ]:
class Node:
    """A node in the decision tree."""
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None, impurity=0.0, n_samples=0):
        self.feature   = feature      # feature index to split on
        self.threshold = threshold    # split threshold
        self.left      = left         # left child node
        self.right     = right        # right child node
        self.value     = value        # leaf prediction (majority class)
        self.impurity  = impurity     # impurity at this node
        self.n_samples = n_samples    # samples reaching this node

    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """
    Binary Decision Tree Classifier built from scratch.
    Supports entropy and gini criteria, max_depth, min_samples_split.
    Also supports feature subsampling (used by Random Forest).
    """

    def __init__(self, max_depth=None, min_samples_split=2,
                 criterion='entropy', max_features=None, random_state=None):
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.criterion        = criterion
        self.max_features     = max_features   # None = use all
        self.random_state     = random_state
        self.root             = None
        self.n_features_      = None
        self._rng             = np.random.RandomState(random_state)

    # ── Impurity function ───────────────────────────────────────────
    def _impurity(self, y):
        return entropy(y) if self.criterion == 'entropy' else gini(y)

    # ── Best split search ───────────────────────────────────────────
    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_gain = -1
        best_feat, best_thresh = None, None

        # Feature subsampling (the key Random Forest trick)
        if self.max_features is None:
            feature_indices = np.arange(n_features)
        elif self.max_features == 'sqrt':
            k = max(1, int(np.sqrt(n_features)))
            feature_indices = self._rng.choice(n_features, k, replace=False)
        elif self.max_features == 'log2':
            k = max(1, int(np.log2(n_features)))
            feature_indices = self._rng.choice(n_features, k, replace=False)
        else:
            k = max(1, int(self.max_features))
            feature_indices = self._rng.choice(n_features, k, replace=False)

        for feat in feature_indices:
            thresholds = np.unique(X[:, feat])
            # Try midpoints between consecutive values
            for i in range(len(thresholds) - 1):
                thresh = (thresholds[i] + thresholds[i+1]) / 2
                left_mask  = X[:, feat] <= thresh
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask], self.criterion)
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh

        return best_feat, best_thresh

    # ── Recursive tree building ─────────────────────────────────────
    def _build(self, X, y, depth=0):
        n_samples = len(y)
        imp = self._impurity(y)

        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) \
           or n_samples < self.min_samples_split \
           or imp == 0.0:
            leaf_value = int(Counter(y.astype(int)).most_common(1)[0][0])
            return Node(value=leaf_value, impurity=imp, n_samples=n_samples)

        feat, thresh = self._best_split(X, y)
        if feat is None:   # no valid split found
            leaf_value = int(Counter(y.astype(int)).most_common(1)[0][0])
            return Node(value=leaf_value, impurity=imp, n_samples=n_samples)

        left_mask  = X[:, feat] <= thresh
        right_mask = ~left_mask

        left  = self._build(X[left_mask],  y[left_mask],  depth + 1)
        right = self._build(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=feat, threshold=thresh,
                    left=left, right=right,
                    impurity=imp, n_samples=n_samples)

    # ── Public API ──────────────────────────────────────────────────
    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.root = self._build(X, y)
        return self

    def _predict_one(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

    def predict_proba(self, X):
        """Return fraction of class 1 at each leaf (used for forest voting)."""
        def _proba(x, node):
            if node.is_leaf():
                return float(node.value)
            if x[node.feature] <= node.threshold:
                return _proba(x, node.left)
            return _proba(x, node.right)
        return np.array([_proba(x, self.root) for x in X])


print('DecisionTreeClassifier defined ✓')

In [ ]:
# ── Test a single decision tree ───────────────────────────────────
# 2D dataset: two moons style, easy to visualise
def make_blobs_2d(n=200):
    X0 = np.random.randn(n//2, 2) + np.array([-1.5, 0])
    X1 = np.random.randn(n//2, 2) + np.array([1.5, 0])
    X  = np.vstack([X0, X1])
    y  = np.array([0]*(n//2) + [1]*(n//2))
    return X, y

X_blob, y_blob = make_blobs_2d(300)

def plot_decision_boundary(model, X, y, ax, title, res=200):
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, res),
                         np.linspace(y_min, y_max, res))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cmap_bg = ListedColormap(['#1a2744', '#1a3322'])
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    colors = [ORANGE if yi == 0 else GREEN for yi in y]
    ax.scatter(X[:,0], X[:,1], c=colors, s=20, edgecolors='none', alpha=0.8)
    acc = np.mean(model.predict(X) == y)
    ax.set_title(f'{title}\nTrain acc: {acc:.3f}', color=BLUE)
    ax.grid(True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, depth, title in zip(axes,
    [2, 5, None],
    ['Shallow tree (depth=2)', 'Medium tree (depth=5)', 'Full tree (no limit)']):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=0)
    dt.fit(X_blob, y_blob)
    plot_decision_boundary(dt, X_blob, y_blob, ax, title)

plt.suptitle('Single Decision Tree — Effect of Max Depth', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()
print('Notice: full tree memorises training data (overfits).')

---
## Part 3 — Visualising a Decision Tree's Splits

Let's draw the actual tree structure to understand how splits are made.

In [ ]:
def draw_tree(node, ax, x=0.5, y=1.0, dx=0.25, dy=0.18,
              feature_names=None, depth=0, max_draw_depth=4):
    """Recursively draw tree nodes."""
    if depth > max_draw_depth:
        return
    if node.is_leaf():
        col = GREEN if node.value == 1 else ORANGE
        label = f'Leaf\nclass={node.value}\nn={node.n_samples}'
    else:
        col = BLUE
        fname = feature_names[node.feature] if feature_names else f'f{node.feature}'
        label = f'{fname}\n≤ {node.threshold:.2f}\nn={node.n_samples}\nIG={node.impurity:.2f}'

    ax.text(x, y, label, ha='center', va='center', fontsize=6.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.3,
                      edgecolor=col, linewidth=1.5),
            color='white')

    if not node.is_leaf() and depth < max_draw_depth:
        lx, rx = x - dx, x + dx
        ny = y - dy
        ax.plot([x, lx], [y - 0.03, ny + 0.03], color='#444', lw=1)
        ax.plot([x, rx], [y - 0.03, ny + 0.03], color='#444', lw=1)
        ax.text((x + lx)/2, (y + ny)/2, 'Yes', fontsize=6, color='#aaa', ha='center')
        ax.text((x + rx)/2, (y + ny)/2, 'No',  fontsize=6, color='#aaa', ha='center')
        draw_tree(node.left,  ax, lx, ny, dx*0.55, dy, feature_names, depth+1, max_draw_depth)
        draw_tree(node.right, ax, rx, ny, dx*0.55, dy, feature_names, depth+1, max_draw_depth)


dt_small = DecisionTreeClassifier(max_depth=3, random_state=0)
dt_small.fit(X_blob, y_blob)

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 1); ax.set_ylim(0.05, 1.1)
ax.axis('off')
draw_tree(dt_small.root, ax, feature_names=['x₁', 'x₂'])
ax.set_title('Decision Tree Structure (depth=3)', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 4 — The Problem with One Tree: Variance

A single deep tree **overfits** — it learns the training data's noise, not the signal.  
If we train on slightly different data, we get a very different tree.

This is called **high variance**.

**Idea**: Train many different trees and let them vote.  
Their errors will cancel out. This is **Bagging** (Bootstrap Aggregation).

In [ ]:
# Show how different samples → different trees
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, ax in enumerate(axes.ravel()):
    # Bootstrap sample (sampling with replacement)
    idx = np.random.choice(len(X_blob), len(X_blob), replace=True)
    Xs, ys = X_blob[idx], y_blob[idx]

    dt_i = DecisionTreeClassifier(max_depth=5, random_state=i)
    dt_i.fit(Xs, ys)
    plot_decision_boundary(dt_i, X_blob, y_blob, ax, f'Tree {i+1} (bootstrap)')

plt.suptitle('8 Trees on Different Bootstrap Samples — Notice the Variance!',
             color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()
print('Each tree is different. The Random Forest averages all of them.')

---
## Part 5 — Bootstrap Sampling

**Bootstrap**: Sample $n$ points **with replacement** from $n$ training points.  
On average, each bootstrap sample contains ~**63.2%** of unique original points.  
The remaining ~36.8% are called **Out-of-Bag (OOB)** samples — free validation set!

$$P(\text{sample not chosen}) = \left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0.368$$

In [ ]:
# Demonstrate bootstrap sampling statistics
n_points = 1000
n_trials = 500
pct_unique = []

for _ in range(n_trials):
    idx = np.random.choice(n_points, n_points, replace=True)
    pct_unique.append(len(np.unique(idx)) / n_points)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram of % unique
axes[0].hist(pct_unique, bins=30, color=GREEN, edgecolor='#0d1117', alpha=0.85)
axes[0].axvline(np.mean(pct_unique), color=GOLD, lw=2, linestyle='--',
                label=f'Mean: {np.mean(pct_unique):.3f}')
axes[0].axvline(1 - 1/np.e, color=ORANGE, lw=2, linestyle=':',
                label=f'Theory: {1-1/np.e:.3f}')
axes[0].set_xlabel('Fraction of unique samples'); axes[0].set_ylabel('Count')
axes[0].set_title('Bootstrap: unique sample fraction\n(~63.2% always)', color=BLUE)
axes[0].legend()

# Show one bootstrap sample visually
idx = np.random.choice(20, 20, replace=True)
oob = np.setdiff1d(np.arange(20), idx)
axes[1].scatter(np.arange(20), np.ones(20), color='#444', s=300, marker='s', label='All samples')
axes[1].scatter(idx, np.ones(len(idx))*1.2, color=GREEN, s=80, label='In bootstrap')
axes[1].scatter(oob, np.ones(len(oob))*0.8, color=ORANGE, s=80, marker='x', label='OOB (free validation)')
axes[1].set_yticks([]); axes[1].set_xlabel('Sample index')
axes[1].set_title(f'One bootstrap sample\n{len(np.unique(idx))}/20 unique, {len(oob)} OOB', color=BLUE)
axes[1].legend(fontsize=8)

# How ensemble accuracy improves with more trees
# Simulate: each tree has accuracy p, voting improves accuracy
from math import comb
def ensemble_acc(p, n_trees):
    """Probability majority vote is correct (binomial majority)."""
    majority = n_trees // 2 + 1
    return sum(comb(n_trees, k) * p**k * (1-p)**(n_trees-k)
               for k in range(majority, n_trees+1))

tree_counts = np.arange(1, 101, 2)
for p, col, lbl in [(0.6, GREEN, 'tree acc=0.60'), (0.7, BLUE, 'tree acc=0.70'),
                    (0.8, PURPLE, 'tree acc=0.80')]:
    acc_vals = [ensemble_acc(p, n) for n in tree_counts]
    axes[2].plot(tree_counts, acc_vals, color=col, lw=2, label=lbl)
axes[2].set_xlabel('Number of trees'); axes[2].set_ylabel('Ensemble accuracy')
axes[2].set_title('Voting improves accuracy\n(law of large numbers)', color=BLUE)
axes[2].legend(); axes[2].grid(True)

plt.suptitle('Bootstrap Aggregation (Bagging) Fundamentals', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 6 — Random Forest: Bagging + Feature Randomness

Bagging alone still produces **correlated trees** — if one feature dominates, all trees use it.

**Random Forest** adds one more trick: at each split, only consider a **random subset of features**.

| Parameter | Typical value |
|-----------|---------------|
| Features per split | $\sqrt{p}$ for classification, $p/3$ for regression |
| Bootstrap | Yes |
| n_estimators | 100–500 |

This **decorrelates** the trees, making the ensemble much stronger.

In [ ]:
class RandomForestClassifier:
    """
    Random Forest = Bagging + Feature subsampling at each split.
    Built entirely from our DecisionTreeClassifier above.
    """

    def __init__(self, n_estimators=100, max_depth=None,
                 min_samples_split=2, max_features='sqrt',
                 criterion='entropy', random_state=None):
        self.n_estimators     = n_estimators
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.max_features     = max_features
        self.criterion        = criterion
        self.random_state     = random_state
        self.trees            = []
        self.oob_indices_     = []     # OOB index sets per tree
        self.feature_importances_ = None

    def fit(self, X, y):
        n, p = X.shape
        self.trees = []
        self.oob_indices_ = []
        oob_votes  = np.zeros((n, 2))   # accumulate OOB predictions

        rng = np.random.RandomState(self.random_state)
        importances = np.zeros(p)

        for i in range(self.n_estimators):
            seed = rng.randint(0, 1_000_000)

            # 1. Bootstrap sample
            boot_idx = rng.choice(n, n, replace=True)
            oob_idx  = np.setdiff1d(np.arange(n), boot_idx)
            self.oob_indices_.append(oob_idx)

            X_boot, y_boot = X[boot_idx], y[boot_idx]

            # 2. Train a tree with feature subsampling
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                max_features=self.max_features,
                random_state=seed
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

            # 3. OOB predictions
            if len(oob_idx) > 0:
                oob_preds = tree.predict(X[oob_idx])
                for j, idx in enumerate(oob_idx):
                    oob_votes[idx, int(oob_preds[j])] += 1

            # 4. Accumulate feature importances (impurity-based)
            importances += self._tree_importance(tree.root, p)

        # OOB accuracy
        oob_mask = oob_votes.sum(axis=1) > 0
        oob_pred = np.argmax(oob_votes[oob_mask], axis=1)
        self.oob_score_ = np.mean(oob_pred == y[oob_mask])

        # Normalise importances
        self.feature_importances_ = importances / importances.sum()

        return self

    def _tree_importance(self, node, n_features):
        """Sum of impurity decreases, weighted by n_samples, per feature."""
        imp = np.zeros(n_features)
        if node.is_leaf():
            return imp
        n = node.n_samples
        n_l = node.left.n_samples
        n_r = node.right.n_samples
        decrease = (n * node.impurity
                    - n_l * node.left.impurity
                    - n_r * node.right.impurity)
        imp[node.feature] += decrease
        imp += self._tree_importance(node.left,  n_features)
        imp += self._tree_importance(node.right, n_features)
        return imp

    def predict(self, X):
        """Majority vote across all trees."""
        votes = np.array([tree.predict(X) for tree in self.trees])  # (T, n)
        return np.apply_along_axis(
            lambda col: Counter(col.astype(int)).most_common(1)[0][0],
            axis=0, arr=votes
        )

    def predict_proba(self, X):
        """Average class probability across all trees."""
        probas = np.array([tree.predict_proba(X) for tree in self.trees])
        return probas.mean(axis=0)


print('RandomForestClassifier defined ✓')

In [ ]:
# ── Train and visualise the Random Forest ─────────────────────────
rf = RandomForestClassifier(n_estimators=50, max_depth=5,
                             max_features='sqrt', random_state=42)
rf.fit(X_blob, y_blob)

dt_full = DecisionTreeClassifier(max_depth=None, random_state=0)
dt_full.fit(X_blob, y_blob)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(dt_full, X_blob, y_blob, axes[0], 'Single full Decision Tree')
plot_decision_boundary(rf,      X_blob, y_blob, axes[1], f'Random Forest (50 trees)\nOOB score: {rf.oob_score_:.3f}')

plt.suptitle('Decision Tree vs Random Forest', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 7 — Out-of-Bag (OOB) Error

Each tree is trained on a bootstrap sample (~63% of data).  
The remaining ~37% (**OOB samples**) act as a **free held-out validation set** for that tree.

We can estimate generalisation error without a separate test set!  
This is powerful — no wasted data.

In [ ]:
# Track OOB error vs number of trees
from copy import deepcopy

n_all = len(X_blob)
oob_votes_accum = np.zeros((n_all, 2))
oob_errors = []

rng2 = np.random.RandomState(42)
all_trees_data = []

for i in range(100):
    boot_idx = rng2.choice(n_all, n_all, replace=True)
    oob_idx  = np.setdiff1d(np.arange(n_all), boot_idx)

    tree_i = DecisionTreeClassifier(max_depth=5, max_features='sqrt', random_state=i)
    tree_i.fit(X_blob[boot_idx], y_blob[boot_idx])

    if len(oob_idx) > 0:
        preds_oob = tree_i.predict(X_blob[oob_idx])
        for j, idx in enumerate(oob_idx):
            oob_votes_accum[idx, int(preds_oob[j])] += 1

    has_votes = oob_votes_accum.sum(axis=1) > 0
    oob_pred  = np.argmax(oob_votes_accum[has_votes], axis=1)
    err = 1 - np.mean(oob_pred == y_blob[has_votes])
    oob_errors.append(err)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, 101), oob_errors, color=GREEN, lw=2, label='OOB error')
ax.axhline(oob_errors[-1], color=GOLD, lw=1.5, linestyle='--',
           label=f'Converged OOB error: {oob_errors[-1]:.3f}')
ax.set_xlabel('Number of trees'); ax.set_ylabel('OOB error rate')
ax.set_title('OOB Error Converges — No Need for Separate Validation Set', color=BLUE)
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

---
## Part 8 — Feature Importance

Random Forests give us **impurity-based feature importance** for free.

At each internal node, a split reduces impurity. We sum those reductions across all trees, weighted by the number of samples reaching each node.

$$\text{Importance}(f) = \frac{1}{T}\sum_{t} \sum_{\text{nodes using } f} n_v \cdot \Delta\text{impurity}_v$$

In [ ]:
# Multi-feature dataset where we know the truth
n3 = 400
# 6 features: only 2 are truly informative
X_multi = np.random.randn(n3, 6)
# Signal comes only from feature 0 and feature 2
signal = 2.0 * X_multi[:, 0] - 1.5 * X_multi[:, 2]
y_multi = (signal + np.random.randn(n3) * 0.5 > 0).astype(int)

feature_names = ['Feature 0\n(strong)', 'Feature 1\n(noise)',
                 'Feature 2\n(moderate)', 'Feature 3\n(noise)',
                 'Feature 4\n(noise)', 'Feature 5\n(noise)']

rf_multi = RandomForestClassifier(n_estimators=100, max_depth=6,
                                   max_features='sqrt', random_state=0)
rf_multi.fit(X_multi, y_multi)

importances = rf_multi.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of importances
bar_colors = [GREEN if i in [0, 2] else '#444' for i in sorted_idx]
axes[0].bar(range(6), importances[sorted_idx], color=bar_colors, edgecolor='#0d1117')
axes[0].set_xticks(range(6))
axes[0].set_xticklabels([feature_names[i] for i in sorted_idx], fontsize=9)
axes[0].set_ylabel('Normalised importance')
axes[0].set_title('Feature Importances\n(green = truly informative)', color=BLUE)
axes[0].grid(True, axis='y')

# Cumulative importance
cum_imp = np.cumsum(importances[sorted_idx])
axes[1].plot(range(1, 7), cum_imp, color=PURPLE, lw=2.5, marker='o', markersize=6)
axes[1].axhline(0.8, color=GOLD, lw=1.5, linestyle='--', label='80% threshold')
axes[1].set_xlabel('Top-k features'); axes[1].set_ylabel('Cumulative importance')
axes[1].set_title('Cumulative Feature Importance\n(feature selection guide)', color=BLUE)
axes[1].legend(); axes[1].grid(True)

plt.suptitle(f'Feature Importance — RF correctly finds informative features\n(OOB={rf_multi.oob_score_:.3f})',
             color=GOLD, fontsize=12)
plt.tight_layout(); plt.show()

for i in sorted_idx:
    print(f'  {feature_names[i].split(chr(10))[0]:<12}: {importances[i]:.4f}')

---
## Part 9 — Hyperparameter Effects

The key hyperparameters and their effect on bias-variance tradeoff:

In [ ]:
# ── How does n_estimators affect accuracy? ────────────────────────
# Use a more complex dataset for better signal
def make_circles(n=400):
    """Two concentric circles — non-linearly separable."""
    theta = np.random.uniform(0, 2*np.pi, n//2)
    X0    = np.column_stack([0.5*np.cos(theta), 0.5*np.sin(theta)]) + np.random.randn(n//2, 2)*0.1
    X1    = np.column_stack([1.5*np.cos(theta), 1.5*np.sin(theta)]) + np.random.randn(n//2, 2)*0.1
    return np.vstack([X0, X1]), np.array([0]*(n//2) + [1]*(n//2))

X_circ, y_circ = make_circles(400)
# Split train/test
split = int(0.75 * len(X_circ))
X_tr, X_te = X_circ[:split], X_circ[split:]
y_tr, y_te = y_circ[:split], y_circ[split:]

n_trees_list = [1, 2, 5, 10, 20, 50, 100, 150]
train_accs, test_accs = [], []

for n_t in n_trees_list:
    rf_t = RandomForestClassifier(n_estimators=n_t, max_depth=8,
                                   max_features='sqrt', random_state=0)
    rf_t.fit(X_tr, y_tr)
    train_accs.append(np.mean(rf_t.predict(X_tr) == y_tr))
    test_accs.append(np.mean(rf_t.predict(X_te) == y_te))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].plot(n_trees_list, train_accs, color=GREEN,  lw=2, marker='o', label='Train')
axes[0].plot(n_trees_list, test_accs,  color=ORANGE, lw=2, marker='s', label='Test')
axes[0].set_xlabel('n_estimators'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Effect of n_estimators\n(more trees = less variance)', color=BLUE)
axes[0].legend(); axes[0].grid(True)

# Effect of max_depth
depths = [1, 2, 3, 5, 8, 12, None]
depth_labels = ['1','2','3','5','8','12','∞']
tr_d, te_d = [], []
for d in depths:
    rf_d = RandomForestClassifier(n_estimators=50, max_depth=d,
                                   max_features='sqrt', random_state=0)
    rf_d.fit(X_tr, y_tr)
    tr_d.append(np.mean(rf_d.predict(X_tr) == y_tr))
    te_d.append(np.mean(rf_d.predict(X_te) == y_te))

x_pos = np.arange(len(depths))
axes[1].plot(x_pos, tr_d, color=GREEN,  lw=2, marker='o', label='Train')
axes[1].plot(x_pos, te_d, color=ORANGE, lw=2, marker='s', label='Test')
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(depth_labels)
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Effect of max_depth\n(sweet spot exists)', color=BLUE)
axes[1].legend(); axes[1].grid(True)

# Decision boundary of final RF on circles
rf_circ = RandomForestClassifier(n_estimators=100, max_depth=8,
                                  max_features='sqrt', random_state=42)
rf_circ.fit(X_tr, y_tr)
plot_decision_boundary(rf_circ, X_circ, y_circ, axes[2],
                       f'RF on concentric circles\nTest acc: {np.mean(rf_circ.predict(X_te)==y_te):.3f}')

plt.suptitle('Hyperparameter Analysis', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 10 — Regression Trees & Random Forest Regressor

The same ideas work for **continuous targets** (regression).

Instead of:
- Impurity: use **Variance** (MSE) instead of Entropy/Gini
- Leaf value: use **mean** of samples instead of majority class
- Aggregation: use **average** predictions instead of majority vote

In [ ]:
class DecisionTreeRegressor:
    """Regression Tree — splits by variance reduction, predicts mean."""

    def __init__(self, max_depth=None, min_samples_split=5,
                 max_features=None, random_state=None):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.root              = None
        self._rng              = np.random.RandomState(random_state)

    def _variance_reduction(self, y_parent, y_left, y_right):
        n = len(y_parent)
        return (np.var(y_parent)
                - (len(y_left)/n) * np.var(y_left)
                - (len(y_right)/n) * np.var(y_right))

    def _best_split(self, X, y):
        n, p = X.shape
        best_gain, best_feat, best_thresh = -np.inf, None, None

        if self.max_features == 'sqrt':
            feats = self._rng.choice(p, max(1, int(np.sqrt(p))), replace=False)
        else:
            feats = np.arange(p)

        for feat in feats:
            for thresh in np.unique(X[:, feat])[:-1]:
                lm = X[:, feat] <= thresh
                if lm.sum() == 0 or (~lm).sum() == 0:
                    continue
                gain = self._variance_reduction(y, y[lm], y[~lm])
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh

    def _build(self, X, y, depth=0):
        if ((self.max_depth and depth >= self.max_depth)
                or len(y) < self.min_samples_split):
            return Node(value=np.mean(y), n_samples=len(y))

        feat, thresh = self._best_split(X, y)
        if feat is None:
            return Node(value=np.mean(y), n_samples=len(y))

        lm = X[:, feat] <= thresh
        return Node(feature=feat, threshold=thresh,
                    left=self._build(X[lm], y[lm], depth+1),
                    right=self._build(X[~lm], y[~lm], depth+1),
                    n_samples=len(y), impurity=np.var(y))

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.root = self._build(X, y)
        return self

    def _pred_one(self, x, node):
        if node.is_leaf(): return node.value
        return self._pred_one(x, node.left if x[node.feature] <= node.threshold else node.right)

    def predict(self, X):
        return np.array([self._pred_one(x, self.root) for x in X])


class RandomForestRegressor:
    """Random Forest for regression — averages predictions of many trees."""

    def __init__(self, n_estimators=50, max_depth=None, min_samples_split=5,
                 max_features='sqrt', random_state=None):
        self.n_estimators     = n_estimators
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.max_features     = max_features
        self.random_state     = random_state
        self.trees            = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n   = len(X)
        self.trees = []
        for i in range(self.n_estimators):
            idx   = rng.choice(n, n, replace=True)
            tree  = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                random_state=rng.randint(0, 1_000_000)
            )
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)
        return self

    def predict(self, X):
        return np.mean([t.predict(X) for t in self.trees], axis=0)


# ── Demo: 1D regression ──────────────────────────────────────────
x_reg = np.linspace(0, 10, 150)
y_reg = np.sin(x_reg) + 0.5*np.cos(2*x_reg) + np.random.randn(150)*0.3
X_reg = x_reg.reshape(-1, 1)

x_test = np.linspace(0, 10, 400).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
configs = [(2, 'Shallow tree (d=2)'), (8, 'Deep tree (d=8)'), (None, 'RF 50 trees')]

for ax, (depth, title) in zip(axes, configs):
    ax.scatter(x_reg, y_reg, color=BLUE, s=15, alpha=0.6, label='Data')
    if 'RF' in title:
        rfr = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42)
        rfr.fit(X_reg, y_reg)
        preds = rfr.predict(x_test)
        r2 = 1 - np.sum((y_reg - rfr.predict(X_reg))**2) / np.sum((y_reg - y_reg.mean())**2)
    else:
        dtr = DecisionTreeRegressor(max_depth=depth, min_samples_split=3, random_state=0)
        dtr.fit(X_reg, y_reg)
        preds = dtr.predict(x_test)
        r2 = 1 - np.sum((y_reg - dtr.predict(X_reg))**2) / np.sum((y_reg - y_reg.mean())**2)

    ax.plot(x_test, preds, color=GREEN, lw=2.5, label='Prediction')
    ax.set_title(f'{title}\nR²={r2:.3f}', color=BLUE)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(); ax.grid(True)

plt.suptitle('Regression Trees — From Underfitting to Smooth RF', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 11 — Prediction Intervals with Random Forest

Since we have many trees, we can estimate **uncertainty** — not just point predictions, but a range.  
The spread of individual tree predictions tells us how confident the forest is.

In [ ]:
rfr2 = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rfr2.fit(X_reg, y_reg)

# Individual tree predictions
x_plot = np.linspace(0, 10, 300).reshape(-1, 1)
tree_preds = np.array([t.predict(x_plot) for t in rfr2.trees])  # (T, n_test)

mean_pred = tree_preds.mean(axis=0)
std_pred  = tree_preds.std(axis=0)

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(x_reg, y_reg, color=BLUE, s=20, alpha=0.7, zorder=5, label='Training data')

# Show a few individual trees
for i in range(8):
    ax.plot(x_plot, tree_preds[i], color='#333', lw=0.8, alpha=0.5)

ax.plot(x_plot, mean_pred, color=GREEN, lw=2.5, label='Mean prediction', zorder=10)
ax.fill_between(x_plot.ravel(),
                mean_pred - 1.96*std_pred,
                mean_pred + 1.96*std_pred,
                alpha=0.2, color=GREEN, label='95% prediction interval')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('RF Prediction Intervals — Uncertainty from Tree Spread', color=BLUE, fontsize=12)
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

---
## Part 12 — Bagging vs Single Tree: Bias-Variance Decomposition

The Random Forest reduces **variance** without increasing **bias** much.  
We can demonstrate this directly by measuring error across many experiments.

In [ ]:
# True function
def true_fn(x): return np.sin(x) + 0.3*x

x_fixed = np.linspace(0, 6, 80)
x_eval  = np.linspace(0, 6, 20)   # evaluate bias/variance here

n_experiments = 30
dt_preds  = np.zeros((n_experiments, len(x_eval)))
rf_preds  = np.zeros((n_experiments, len(x_eval)))

for exp in range(n_experiments):
    x_train = np.random.uniform(0, 6, 60)
    y_train = true_fn(x_train) + np.random.randn(60) * 0.4
    X_train = x_train.reshape(-1,1)
    X_eval  = x_eval.reshape(-1,1)

    dt_exp = DecisionTreeRegressor(max_depth=None, min_samples_split=2, random_state=exp)
    dt_exp.fit(X_train, y_train)
    dt_preds[exp] = dt_exp.predict(X_eval)

    rf_exp = RandomForestRegressor(n_estimators=30, max_depth=8, random_state=exp)
    rf_exp.fit(X_train, y_train)
    rf_preds[exp] = rf_exp.predict(X_eval)

true_y = true_fn(x_eval)

dt_bias2 = (dt_preds.mean(axis=0) - true_y)**2
dt_var   = dt_preds.var(axis=0)
rf_bias2 = (rf_preds.mean(axis=0) - true_y)**2
rf_var   = rf_preds.var(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Spread of individual predictions
for preds, col, lbl in [(dt_preds, ORANGE, 'Single Trees'), (rf_preds, GREEN, 'Random Forest')]:
    for exp in range(n_experiments):
        axes[0].plot(x_eval, preds[exp], color=col, lw=0.5, alpha=0.3)
axes[0].plot(x_eval, true_y, color='white', lw=2, label='True fn', zorder=10)
axes[0].set_title('Prediction spread across 30 experiments', color=BLUE)
axes[0].legend([mpatches.Patch(color=ORANGE), mpatches.Patch(color=GREEN),
                mpatches.Patch(color='white')],
               ['Trees', 'RF', 'True'], fontsize=8)
axes[0].grid(True)

# Bias²
axes[1].plot(x_eval, dt_bias2, color=ORANGE, lw=2, label='Single tree')
axes[1].plot(x_eval, rf_bias2, color=GREEN,  lw=2, label='Random Forest')
axes[1].set_title(f'Bias²  (tree={dt_bias2.mean():.3f}, RF={rf_bias2.mean():.3f})', color=BLUE)
axes[1].set_xlabel('x'); axes[1].legend(); axes[1].grid(True)

# Variance
axes[2].plot(x_eval, dt_var, color=ORANGE, lw=2, label='Single tree')
axes[2].plot(x_eval, rf_var, color=GREEN,  lw=2, label='Random Forest')
axes[2].set_title(f'Variance  (tree={dt_var.mean():.3f}, RF={rf_var.mean():.3f})', color=BLUE)
axes[2].set_xlabel('x'); axes[2].legend(); axes[2].grid(True)

plt.suptitle('Bias-Variance Decomposition — RF dramatically cuts variance', color=GOLD, fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 13 — Summary

```
┌──────────────────────────────────────────────────────────────────┐
│              RANDOM FOREST — CHEAT SHEET                         │
├───────────────────────┬──────────────────────────────────────────┤
│ Impurity (classif.)   │ Entropy or Gini                          │
│ Impurity (regression) │ Variance (MSE)                           │
│ Leaf value (classif.) │ Majority class                           │
│ Leaf value (regress.) │ Mean of samples                          │
│ Aggregation           │ Majority vote / Average                  │
├───────────────────────┼──────────────────────────────────────────┤
│ Key trick 1           │ Bootstrap each tree (bagging)            │
│ Key trick 2           │ Random feature subset at each split      │
│ OOB score             │ Free validation — no separate set needed │
│ Feature importance    │ Weighted impurity decrease across trees  │
├───────────────────────┼──────────────────────────────────────────┤
│ n_estimators          │ 100–500; more = less variance            │
│ max_depth             │ None or 8–20; deeper = more variance     │
│ max_features          │ sqrt(p) classif, p/3 regression          │
│ min_samples_split     │ 2–10; higher = less overfitting          │
└───────────────────────┴──────────────────────────────────────────┘

Bias-Variance: RF ≈ low bias (deep trees), low variance (averaging)
```

In [ ]:
print('''
╔═══════════════════════════════════════════════════════════════════╗
║          Notebook complete! What you built from scratch:          ║
╠═══════════════════════════════════════════════════════════════════╣
║  ✅ Entropy, Gini impurity, Information Gain                      ║
║  ✅ Decision Tree (split search, recursive build, predict)        ║
║  ✅ Tree visualiser (structure diagram)                           ║
║  ✅ Bootstrap sampling + OOB error estimation                     ║
║  ✅ Random Forest Classifier (bagging + feature subsampling)      ║
║  ✅ Feature importances (impurity-based)                          ║
║  ✅ Hyperparameter analysis (n_estimators, max_depth)             ║
║  ✅ Regression Tree + Random Forest Regressor                     ║
║  ✅ Prediction intervals from tree spread                         ║
║  ✅ Bias-Variance decomposition proof via simulation              ║
╚═══════════════════════════════════════════════════════════════════╝
''')